In [1]:
import gc
import os
import sys

import numpy as np
import optuna
import pandas as pd
import torch
from sklearn.model_selection import KFold

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
%load_ext autoreload
%autoreload 2
from drGAT import No_atten_drGAT
from drGAT.load_data import load_data
from drGAT.sampler import RandomSampler

In [3]:
drugAct, pos_num, null_mask, S_d, S_c, S_g, A_cg, A_dg = load_data("gdsc1")

load gdsc1
unique drugs: 73
unique genes: 166
DTI unique genes:  166
Top 90% variable genes:  1957
Total:  2099
Final gene exp shape: (916, 2099)
Final drug Act shape: (331, 916)


100%|██████████| 25/25 [00:02<00:00, 12.33it/s]


Done!


In [4]:
PATH = "../gdsc1_data/"

In [5]:
def objective(trial):
    params = {
        "n_drug": S_d.shape[0],
        "n_cell": S_c.shape[0],
        "n_gene": S_g.shape[0],
        "dropout1": trial.suggest_categorical("dropout1", [0.1, 0.2, 0.3, 0.4, 0.5]),
        "dropout2": trial.suggest_categorical("dropout2", [0.1, 0.2, 0.3, 0.4, 0.5]),
        "hidden1": trial.suggest_categorical(
            "hidden1",
            [256, 512, 1028],
        ),
        "hidden2": trial.suggest_categorical(
            "hidden2",
            [
                128,
                256,
                512,
            ],
        ),
        "hidden3": trial.suggest_categorical(
            "hidden3",
            [
                64,
                128,
                256,
            ],
        ),
        "epochs": trial.suggest_categorical("epochs", [10, 50, 100, 200, 500]),
        # "epochs": 2,
        "heads": trial.suggest_categorical("heads", [1, 2, 3, 4, 5]),
        "activation": trial.suggest_categorical(
            "activation", ["relu", "gelu", "swish"]
        ),
        "optimizer": trial.suggest_categorical("optimizer", ["Adam", "AdamW", "SGD"]),
        "lr": trial.suggest_float("lr", 1e-5, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
        "scheduler": trial.suggest_categorical(
            "scheduler", [None, "Cosine", "Step", "Plateau"]
        ),
        "gnn_layer": trial.suggest_categorical(
            "gnn_layer",
            ["GCN", "MPNN"],
        ),
    }

    # スケジューラ関連パラメータの条件付き追加
    if params["scheduler"] == "Cosine":
        params["T_max"] = trial.suggest_int("T_max", 20, 50)
    elif params["scheduler"] == "Step":
        params["scheduler_gamma"] = trial.suggest_float("gamma_step", 0.1, 0.95)
        params["step_size"] = trial.suggest_int("step_size", 10, 30)
    elif params["scheduler"] == "Plateau":
        params["patience"] = trial.suggest_int("patience_plateau", 3, 10)
        params["threshold"] = trial.suggest_float(
            "thresh_plateau", 1e-4, 1e-2, log=True
        )

    if params["hidden1"] < params["hidden2"] or params["hidden2"] < params["hidden3"]:
        raise optuna.TrialPruned("Invalid layer size configuration")

    if params["optimizer"] in ["Adam", "AdamW"]:
        params["amsgrad"] = trial.suggest_categorical("amsgrad", [True, False])

    if params["optimizer"] == "SGD":
        params["momentum"] = trial.suggest_float("momentum", 0.8, 0.95)
        params["nesterov"] = trial.suggest_categorical("nesterov", [True, False])

    # 隠れ層サイズとバッチサイズの関係を制約
    if (params["hidden1"] > 512) and (params["hidden2"] > 256):
        raise optuna.TrialPruned("Memory intensive configuration")

    try:
        k = 5
        kfold = KFold(n_splits=k, shuffle=True, random_state=42)

        res = pd.DataFrame()
        for train_index, test_index in kfold.split(np.arange(pos_num)):
            sampler = RandomSampler(
                drugAct,
                train_index,
                test_index,
                null_mask,
                S_d,
                S_c,
                S_g,
                A_cg,
                A_dg,
                PATH,
            )
            _, best_metrics, _ = No_atten_drGAT.train(
                sampler, params=params, device=device, verbose=False
            )

            res = pd.concat(
                [
                    res,
                    pd.DataFrame(best_metrics, index=["acc", "f1", "auroc", "aupr"]).T,
                ]
            )

        return [float(i) for i in res.mean()]

    except RuntimeError as e:
        if "CUDA out of memory" in str(e):
            print(f"Pruned trial {trial.number}: CUDA OOM")

            # メモリ解放処理
            with torch.cuda.device("cuda"):
                torch.cuda.empty_cache()
            gc.collect()

            # Pruning通知
            raise optuna.TrialPruned(f"OOM at trial {trial.number}")

        else:
            raise e

In [ ]:
name = "GDSC1"
study = optuna.create_study(
    directions=["maximize"] * 4,
    sampler=optuna.samplers.TPESampler(),
    pruner=optuna.pruners.HyperbandPruner(),
    storage="sqlite:///{}_{}.sqlite3".format(name, "GCN_MPNN"),
    study_name=name,
    load_if_exists=True,
)
study.optimize(objective, n_trials=100, n_jobs=8)

[I 2025-03-28 13:24:02,203] A new study created in RDB with name: GDSC1


Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda


  0%|          | 0/100 [00:00<?, ?it/s]

Using device: cuda
Using device: cuda



  0%|          | 0/200 [00:00<?, ?it/s]


  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]



  0%|          | 0/100 [00:00<?, ?it/s]




  0%|          | 0/500 [00:00<?, ?it/s]

...




  0%|          | 0/100 [00:00<?, ?it/s] 1.65s/it]




 44%|████▍     | 219/500 [05:05<04:54,  1.05s/it]

100%|██████████| 100/100 [02:15<00:00,  1.36s/it]


Best model found at epoch 100








 38%|███▊      | 19/50 [00:39<00:54,  1.76s/it]



 64%|██████▍   | 32/50 [00:41<00:21,  1.21s/it]




  1%|          | 1/100 [00:01<02:18,  1.40s/it]s]



 66%|██████▌   | 33/50 [00:42<00:20,  1.22s/it]


 68%|██████▊   | 137/200 [05:07<01:50,  1.75s/it]




  2%|▏         | 2/100 [00:02<01:43,  1.05s/it]t]





 40%|████      | 20/50 [00:42<00:58,  1.94s/it]




 44%|████▍     | 222/500 [05:08<04:55,  1.06s/it]



 69%|██████▉   | 138/200 [05:09<01:47,  1.74s/it]


 66%|██████▋   | 133/200 [05:09<01:53,  1.69s/it]




  3%|▎         | 3/100 [00:03<02:01,  1.26s/it]s]



 70%|███████   | 35/50 [00:45<00:18,  1.26s/it]





  4%|▍         | 4/100 [00:04<01:56,  1.21s/it]




 45%|████▍     | 224/500 [05:10<04:44,  1.03s/it]



 70%|██████▉   | 139/200 [05:11<01:52,  1.84s/it]


 67%|██████▋   | 134/200 [05:11<02:07,  1.93s/it]




  5%|▌         | 5/100 [00:06<02:01,  1.27s/it]t]





 44%|████▍     | 22/50 [00:45<00:52,  1.88s/it]



 74%|███████▍  | 37/50 [00:47<00:15,  1.21s/i

Using device: cuda






 78%|███████▊  | 39/50 [00:50<00:13,  1.24s/it]

  0%|          | 0/100 [00:00<?, ?it/s]


 70%|███████   | 141/200 [05:15<01:58,  2.00s/it]




  8%|▊         | 8/100 [00:10<02:01,  1.32s/it]t]





 48%|████▊     | 24/50 [00:49<00:50,  1.95s/it]



 80%|████████  | 40/50 [00:51<00:12,  1.26s/it]/data/inouey2/conda/envs/genex/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:216: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


  1%|          | 1/100 [00:00<01:33,  1.06it/s]




  9%|▉         | 9/100 [00:11<02:01,  1.33s/it]t]


 68%|██████▊   | 137/200 [05:17<02:04,  1.97s/it]

  2%|▏         | 2/100 [00:02<01:50,  1.1

## Eval

In [ ]:
# test_drug = test_data.values[:, 0]
# test_cell = test_data.values[:, 1]

# test_labels = np.load("data/test_labels.npy")
# test_labels = torch.tensor(test_labels).float()
# test = [drug, cell, gene, edge_index, test_drug, test_cell, test_labels]

In [ ]:
# prob, res, test_attention = drGAT.eval(model, test)